In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/digit-recognizer/sample_submission.csv
/kaggle/input/digit-recognizer/train.csv
/kaggle/input/digit-recognizer/test.csv


In [2]:
import tensorflow as tf
import tensorflow_decision_forests as tfdf

from sklearn.preprocessing import MinMaxScaler

from keras.utils.np_utils import to_categorical
from keras.models import Sequential
from keras.layers import Dense, Flatten, Conv2D, MaxPool2D, BatchNormalization, Dropout
from keras.optimizers import SGD

/opt/conda/lib/python3.10/site-packages/tensorflow_io/python/ops/__init__.py:98: UserWarning: unable to load libtensorflow_io_plugins.so: unable to open file: libtensorflow_io_plugins.so, from paths: ['/opt/conda/lib/python3.10/site-packages/tensorflow_io/python/ops/libtensorflow_io_plugins.so']
caused by: ['/opt/conda/lib/python3.10/site-packages/tensorflow_io/python/ops/libtensorflow_io_plugins.so: undefined symbol: _ZN3tsl6StatusC1EN10tensorflow5error4CodeESt17basic_string_viewIcSt11char_traitsIcEENS_14SourceLocationE']
  warnings.warn(f"unable to load libtensorflow_io_plugins.so: {e}")
/opt/conda/lib/python3.10/site-packages/tensorflow_io/python/ops/__init__.py:104: UserWarning: file system plugins are not loaded: unable to open file: libtensorflow_io.so, from paths: ['/opt/conda/lib/python3.10/site-packages/tensorflow_io/python/ops/libtensorflow_io.so']
caused by: ['/opt/conda/lib/python3.10/site-packages/tensorflow_io/python/ops/libtensorflow_io.so: undefined symbol: _ZTVN10tenso

In [3]:
train_data = pd.read_csv('/kaggle/input/digit-recognizer/train.csv')
test_data = pd.read_csv('/kaggle/input/digit-recognizer/test.csv')

In [4]:
train_data.head()

,label,pixel0,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,...,pixel774,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783
0,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,4,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [5]:
x_train = train_data.drop(['label'], axis=1)
y_train = train_data['label']
x_test = test_data
scaler = MinMaxScaler()

In [6]:
y_train.value_counts()

1    4684
7    4401
3    4351
9    4188
2    4177
6    4137
0    4132
4    4072
8    4063
5    3795
Name: label, dtype: int64

In [7]:
x_train = pd.DataFrame(scaler.fit_transform(x_train), columns=scaler.feature_names_in_)
x_test = pd.DataFrame(scaler.transform(x_test), columns=scaler.feature_names_in_)

x_train = x_train.values.reshape(-1,28,28,1)
x_test = x_test.values.reshape(-1,28,28,1)
y_train = to_categorical(y_train, num_classes=10)

In [8]:
model = Sequential()

model.add(Conv2D(filters=32, kernel_size=(5,5), kernel_initializer='he_uniform', padding='Same', activation='relu', input_shape=(28,28,1)))
model.add(BatchNormalization())
model.add(MaxPool2D(pool_size=(2,2)))

model.add(Conv2D(filters=64, kernel_size=(5,5), kernel_initializer='he_uniform', padding='Same', activation='relu'))
model.add(BatchNormalization())
model.add(MaxPool2D(pool_size=(2,2)))

model.add(Conv2D(filters=128, kernel_size=(5,5), kernel_initializer='he_uniform', padding='Same', activation='relu'))
model.add(BatchNormalization())
model.add(MaxPool2D(pool_size=(2,2)))


model.add(Flatten())
model.add(Dense(128, kernel_initializer='he_uniform', activation='relu'))
model.add(BatchNormalization())


model.add(Dense(10, activation='softmax'))

model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d (Conv2D)             (None, 28, 28, 32)        832       
                                                                 
 batch_normalization (BatchN  (None, 28, 28, 32)       128       
 ormalization)                                                   
                                                                 
 max_pooling2d (MaxPooling2D  (None, 14, 14, 32)       0         
 )                                                               
                                                                 
 conv2d_1 (Conv2D)           (None, 14, 14, 64)        51264     
                                                                 
 batch_normalization_1 (Batc  (None, 14, 14, 64)       256       
 hNormalization)                                                 
                                                        

In [9]:
model.compile(optimizer=SGD(learning_rate=0.01, momentum=0.9,), loss='categorical_crossentropy', metrics=['accuracy'])
model.fit(x_train, y_train, validation_split=0.1, batch_size=50, epochs=59, verbose=2)

predictions = model.predict(x_test)
n_predictions = np.argmax(predictions, axis=1)

Epoch 1/59
756/756 - 96s - loss: 0.1025 - accuracy: 0.9679 - val_loss: 0.0630 - val_accuracy: 0.9807 - 96s/epoch - 127ms/step
Epoch 2/59
756/756 - 96s - loss: 0.0302 - accuracy: 0.9912 - val_loss: 0.0353 - val_accuracy: 0.9895 - 96s/epoch - 127ms/step
Epoch 3/59
756/756 - 94s - loss: 0.0182 - accuracy: 0.9948 - val_loss: 0.0400 - val_accuracy: 0.9867 - 94s/epoch - 124ms/step
Epoch 4/59
756/756 - 95s - loss: 0.0099 - accuracy: 0.9974 - val_loss: 0.0301 - val_accuracy: 0.9907 - 95s/epoch - 125ms/step
Epoch 5/59
756/756 - 95s - loss: 0.0056 - accuracy: 0.9989 - val_loss: 0.0254 - val_accuracy: 0.9924 - 95s/epoch - 125ms/step
Epoch 6/59
756/756 - 95s - loss: 0.0027 - accuracy: 0.9998 - val_loss: 0.0272 - val_accuracy: 0.9917 - 95s/epoch - 125ms/step
Epoch 7/59
756/756 - 95s - loss: 0.0017 - accuracy: 0.9998 - val_loss: 0.0247 - val_accuracy: 0.9919 - 95s/epoch - 125ms/step
Epoch 8/59
756/756 - 93s - loss: 0.0013 - accuracy: 0.9999 - val_loss: 0.0257 - val_accuracy: 0.9924 - 93s/epoch - 123

In [10]:
pd.read_csv('/kaggle/input/digit-recognizer/sample_submission.csv')

,ImageId,Label
0,1,0
1,2,0
2,3,0
3,4,0
4,5,0
...,...,...
27995,27996,0
27996,27997,0
27997,27998,0
27998,27999,0


In [11]:
df = pd.DataFrame()
df['ImageId'] = range(1, 28001)
df['Label'] = n_predictions
df.to_csv('submission.csv', index=False)